In [1]:
# ============================================================
# 🤖 AI CUSTOMER SUPPORT CHATBOT
# Google Colab - Single Cell Application
# ============================================================

!pip -q install gradio transformers torch

import gradio as gr
from transformers import pipeline

# ------------------------------------------------------------
# Load AI chatbot model
# ------------------------------------------------------------

chatbot_model = pipeline(
    "text-generation",
    model="distilgpt2"
)

# ------------------------------------------------------------
# Chatbot response function
# ------------------------------------------------------------

def chatbot_response(message, history):

    if not message.strip():
        return "Please enter a message."

    # Convert previous conversation into text
    conversation = ""

    for user_msg, bot_msg in history:
        conversation += f"Customer: {user_msg}\n"
        conversation += f"Assistant: {bot_msg}\n"

    prompt = (
        conversation +
        f"Customer: {message}\n"
        "Assistant:"
    )

    # Generate response
    result = chatbot_model(
        prompt,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    response = result[0]["generated_text"]

    # Extract assistant response
    if "Assistant:" in response:
        response = response.split("Assistant:")[-1]

    # Stop unwanted continuation
    if "Customer:" in response:
        response = response.split("Customer:")[0]

    response = response.strip()

    if not response:
        response = "I'm sorry, I couldn't understand that. Please try again."

    return response


# ------------------------------------------------------------
# Chatbot Interface
# ------------------------------------------------------------

with gr.Blocks(
    title="AI Customer Support Chatbot"
) as app:

    gr.Markdown(
        """
        # 🤖 AI Customer Support Chatbot

        Welcome! Ask me questions about orders, delivery,
        payments, products, refunds, and customer support.
        """
    )

    chatbot = gr.Chatbot(
        label="Customer Support",
        height=500
    )

    message = gr.Textbox(
        label="Your Message",
        placeholder="Type your question here...",
        lines=2
    )

    with gr.Row():

        send = gr.Button(
            "📤 Send",
            variant="primary"
        )

        clear = gr.Button(
            "🗑️ Clear"
        )

    gr.Markdown(
        """
        ### Example Questions

        • Where is my order?

        • How can I get a refund?

        • My product arrived damaged.

        • How can I cancel my order?

        • I have a payment problem.
        """
    )

    # Send message
    def respond(message, history):

        response = chatbot_response(message, history)

        history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": response}
        ]

        return "", history

    send.click(
        respond,
        inputs=[message, chatbot],
        outputs=[message, chatbot]
    )

    message.submit(
        respond,
        inputs=[message, chatbot],
        outputs=[message, chatbot]
    )

    clear.click(
        lambda: [],
        outputs=chatbot
    )


# ------------------------------------------------------------
# Launch Application
# ------------------------------------------------------------

app.launch(share=True)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://40ebe2e2fc0d1d5aab.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
